# IDG — PPO: all four pairings, 10-seed retrains (1000 iterations)

Each tied-best config is retrained across 10 seeds in both of the pairings it applies to:
- `learned_proposer × perfect_validator` and `learned_proposer × always_approve`
- `perfect_proposer × learned_validator` and `random_proposer × learned_validator`

Runs resume per seed from Google Drive, and the best-scoring seed's checkpoint is saved.

## 0. Setup (run every session): clone the repo, install dependencies, mount Drive, and show current progress

In [ ]:
# Clone the repo onto the Colab VM. Look for a GitHub token in an env var, then a Colab secret, then a hidden prompt.
import os, getpass
REPO_DIR  = "/content/intelligent_disobedience_leader_follower_grid"
REPO_HOST = os.environ.get("REPO_HOST", "github.com/ORGANIZATION/intelligent_disobedience_leader_follower_grid.git")

def _github_pat():
    t = os.environ.get("GITHUB_PAT")
    if t:
        return t
    try:
        from google.colab import userdata
        t = userdata.get("GITHUB_PAT")
        if t:
            return t
    except Exception:
        pass
    return getpass.getpass("GitHub PAT (input hidden): ").strip()

if not os.path.isdir(REPO_DIR):
    _tok = _github_pat()
    !git clone https://{_tok}@{REPO_HOST}
    del _tok
else:
    print("repo already cloned")
%cd /content/intelligent_disobedience_leader_follower_grid
!git pull


In [ ]:
%cd /content/intelligent_disobedience_leader_follower_grid
!pip install -q -r requirements.txt
!pip install -q tensorboardX "ray[tune]" lz4


In [ ]:
%cd /content/intelligent_disobedience_leader_follower_grid
import os, glob, json, shutil, subprocess, tempfile, time
from datetime import datetime
from google.colab import drive

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ.setdefault("RAY_DISABLE_METRICS_COLLECTION", "1")

ALGO = "ppo"
SEEDS = list(range(10))     # 10 seeds per config and pairing

# Map each of the four IDG pairings to the experiment name used to look up its final params.
EXP_NAME = {
    "learned_proposer": f"{ALGO}_learned_proposer_perfect_validator__proposer_sees_lava_False",
    "perfect_proposer": f"{ALGO}_perfect_proposer_learned_validator__proposer_sees_lava_False",
    "always_approve":   f"{ALGO}_learned_proposer_always_approve_validator__proposer_sees_lava_False",
    "random_proposer":  f"{ALGO}_random_proposer_learned_validator__proposer_sees_lava_False",
}


CONFIGS = json.loads(r"""
{
 "prop_jun29_94g": {
  "role": "proposer",
  "iters": 1000,
  "pairings": [
   "learned_proposer",
   "always_approve"
  ],
  "params": {
   "lr": 0.000149243,
   "entropy_coeff": 0.0236241,
   "clip_param": 0.34367,
   "num_epochs": 30
  }
 },
 "prop_jun28_94g": {
  "role": "proposer",
  "iters": 1000,
  "pairings": [
   "learned_proposer",
   "always_approve"
  ],
  "params": {
   "lr": 0.000106692,
   "entropy_coeff": 0.0298724,
   "clip_param": 0.318452,
   "num_epochs": 30
  }
 },
 "prop_jul07_94g": {
  "role": "proposer",
  "iters": 1000,
  "pairings": [
   "learned_proposer",
   "always_approve"
  ],
  "params": {
   "lr": 0.000157056,
   "entropy_coeff": 0.0340401,
   "clip_param": 0.374613,
   "num_epochs": 30
  }
 },
 "val_logstune_100g": {
  "role": "validator",
  "iters": 1000,
  "pairings": [
   "perfect_proposer",
   "random_proposer"
  ],
  "params": {
   "lr": 0.000277978,
   "gamma": 0.94737,
   "entropy_coeff": 0.0095544,
   "clip_param": 0.199534,
   "num_epochs": 15
  }
 },
 "val_jun29_100g": {
  "role": "validator",
  "iters": 1000,
  "pairings": [
   "perfect_proposer",
   "random_proposer"
  ],
  "params": {
   "lr": 0.000290913,
   "gamma": 0.946523,
   "entropy_coeff": 0.00632587,
   "clip_param": 0.203138,
   "num_epochs": 15
  }
 },
 "val_jun30_100g": {
  "role": "validator",
  "iters": 1000,
  "pairings": [
   "perfect_proposer",
   "random_proposer"
  ],
  "params": {
   "lr": 0.000240694,
   "gamma": 0.95186,
   "entropy_coeff": 0.00731704,
   "clip_param": 0.170709,
   "num_epochs": 15
  }
 }
}
""")


drive.mount("/content/drive", force_remount=True)
DRIVE_OUT = f"/content/drive/MyDrive/idg_final_{ALGO}_1000iters"

def _flush():
    try:
        drive.flush_and_unmount()
        drive.mount("/content/drive", force_remount=True)
    except Exception as e:
        print("warn: drive flush/remount:", e)

os.makedirs(DRIVE_OUT, exist_ok=True)
_flush()
os.makedirs(DRIVE_OUT, exist_ok=True)

def csv_name(tag, pairing, seed):
    return f"final_{ALGO}_{tag}_{pairing}_i{CONFIGS[tag]['iters']}_seed{seed}.csv"

def seed_csvs(tag, pairing):
    it = CONFIGS[tag]["iters"]
    return sorted(set(glob.glob(f"{DRIVE_OUT}/final_{ALGO}_{tag}_{pairing}_i{it}_seed*.csv")))

def done_seeds(tag, pairing):
    got = set()
    for f in seed_csvs(tag, pairing):
        try:
            got.add(int(os.path.basename(f).rsplit("seed", 1)[1].split(".")[0]))
        except ValueError:
            pass
    return got

# ---- best-of-seeds checkpoint tracking (per config x pairing) ----
def best_ckpt_paths(tag, pairing):
    stem = f"final_{ALGO}_{tag}_{pairing}_i{CONFIGS[tag]['iters']}_bestckpt"
    return f"{DRIVE_OUT}/{stem}.tar", f"{DRIVE_OUT}/{stem}.json"

def read_best_ckpt(tag, pairing):
    _, jp = best_ckpt_paths(tag, pairing)
    if os.path.exists(jp):
        try:
            return json.load(open(jp))
        except Exception:
            pass
    return {"seed": None, "goal_pct": -1.0}

def maybe_save_best_ckpt(tag, pairing, seed, goal_pct, ckpt_dir):
    """Save this seed's checkpoint to Drive only if its goal% beats the best recorded so far.
    The record is updated only after the tar file is confirmed written."""
    if not os.path.isdir(ckpt_dir):
        print(f"  [{tag}/{pairing} seed {seed}] no checkpoint at {ckpt_dir} -- best-ckpt skipped")
        return False
    if goal_pct <= read_best_ckpt(tag, pairing)["goal_pct"]:
        return False
    tarp, jp = best_ckpt_paths(tag, pairing)
    fd, tmp = tempfile.mkstemp(suffix=".tar")
    os.close(fd)
    try:
        r1 = subprocess.run(["tar", "-cf", tmp, "-C", os.path.dirname(ckpt_dir), os.path.basename(ckpt_dir)])
        r2 = subprocess.run(["cp", "-f", tmp, tarp]) if r1.returncode == 0 else None
    finally:
        os.remove(tmp)
    if r1.returncode != 0 or r2 is None or r2.returncode != 0 or not os.path.exists(tarp):
        print(f"  [{tag}/{pairing} seed {seed}] best-ckpt tar/copy FAILED -- record NOT updated")
        return False
    with open(jp, "w") as f:
        json.dump({"tag": tag, "pairing": pairing, "seed": seed, "goal_pct": goal_pct,
                   "iters": CONFIGS[tag]["iters"], "saved": datetime.now().strftime("%Y%m%d_%H%M%S")}, f)
    return True

_total = sum(len(c["pairings"]) for c in CONFIGS.values()) * len(SEEDS)
print(f"ALGO: {ALGO} | {len(CONFIGS)} configs x their pairings x {len(SEEDS)} seeds = {_total} runs (1000 iters)")
for tag, c in CONFIGS.items():
    for pairing in c["pairings"]:
        b = read_best_ckpt(tag, pairing)
        print(f"  {tag:34} {pairing:18} done: {sorted(done_seeds(tag, pairing)) or '-'}"
              f"  best_ckpt: seed {b['seed']} @ {b['goal_pct']}")
print("Drive:", DRIVE_OUT)


## 1. Train every config in every pairing (resumable). The best seed's checkpoint is saved to the google Drive

In [ ]:
# Create a local logs/tune_custom directory to hold the final summary JSONs for each config x pairing.
os.makedirs("logs/tune_custom", exist_ok=True)

def _csv_goal(path, pairing):
    import csv as _csv
    for r in _csv.DictReader(open(path)):
        if r["pairing"] == pairing:
            return float(r["goal_pct"])
    return None

for tag, cfg in CONFIGS.items():
    for pairing in cfg["pairings"]:
        exp = EXP_NAME[pairing]
        summ_path = f"logs/tune_custom/final_{tag}_{pairing}.json"
        with open(summ_path, "w") as f:
            json.dump([{"experiment": exp, "final_params": cfg["params"]}], f)

        already = done_seeds(tag, pairing)
        todo = [s for s in SEEDS if s not in already]
        if not todo:
            print(f"[{tag} / {pairing}] all {len(SEEDS)} seeds already on Drive -- skip")
            continue
        print(f"\n##### {tag} ({pairing}, {cfg['iters']} iters) -- seeds to run: {todo} #####")
        for k in todo:
            t0 = time.time()
            !rm -f eval_results/seed_runs_perseed_*.csv
            !rm -rf logs/seeds
            !python run_seeds.py --algos {ALGO} --pairings {pairing} --seeds {k} --iters {cfg["iters"]} --summary "{summ_path}"
            csvs = sorted(glob.glob("eval_results/seed_runs_perseed_*.csv"))
            if not csvs:
                print(f"  [{tag}/{pairing} seed {k}] NO CSV produced -- not marked done")
                continue
            goal = _csv_goal(csvs[-1], pairing)
            ckpt_dir = f"logs/seeds/{exp}/seed_{k}"
            new_best = maybe_save_best_ckpt(tag, pairing, k, goal if goal is not None else -1.0, ckpt_dir)
            dst = f"{DRIVE_OUT}/{csv_name(tag, pairing, k)}"
            subprocess.run(["cp", "-f", csvs[-1], dst])
            _flush()
            print(f"  [{tag}/{pairing} seed {k}] {(time.time()-t0)/60:.1f} min  goal={goal}"
                  + ("  ** NEW BEST -> ckpt tarred to Drive" if new_best else "") + f"  -> {dst}")


## 2. Aggregate results into a per-(config × pairing) table (mean ± std, plus the best checkpoint)

In [ ]:
# Aggregate every per-seed CSV on Drive into a per-(config, pairing) table (mean ± std, best checkpoint).
import pandas as pd
rows = []
for tag, cfg in CONFIGS.items():
    for pairing in cfg["pairings"]:
        for f in seed_csvs(tag, pairing):
            d = pd.read_csv(f)
            d = d[d.pairing == pairing].copy()
            d["config_tag"] = tag
            d["pairing_key"] = pairing
            d["iters"] = cfg["iters"]
            rows.append(d)
if not rows:
    raise SystemExit("no per-seed CSVs on Drive yet")
df = pd.concat(rows, ignore_index=True)

summ = []
for (tag, pairing), g in df.groupby(["config_tag", "pairing_key"]):
    v = g["goal_pct"]
    b = read_best_ckpt(tag, pairing)
    summ.append(dict(config=tag, pairing=pairing, iters=int(g["iters"].iloc[0]),
                     n_seeds=g["seed"].nunique(),
                     mean=round(v.mean(), 2), std=round(v.std(ddof=1), 2),
                     min=round(v.min(), 1), max=round(v.max(), 1),
                     best_ckpt_seed=b["seed"], best_ckpt_goal=b["goal_pct"],
                     per_seed=",".join(f"{x:.0f}" for x in sorted(v))))
summary = pd.DataFrame(summ).sort_values(["pairing", "mean"], ascending=[True, False])
display(summary)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
summary.to_csv(f"{DRIVE_OUT}/final_{ALGO}_summary_{ts}.csv", index=False)
with open(f"{DRIVE_OUT}/final_{ALGO}_summary_{ts}.txt", "w") as f:
    f.write(f"final all-pairings 10-seed retrains -- {ALGO} (uniform 1000 iters, goal%)\n")
    f.write(summary.to_string(index=False) + "\n")
_flush()
print("saved summary csv+txt ->", DRIVE_OUT)
